In [2]:
import pandas as pd
import numpy as np
import pickle
import json
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble import StackingClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from scipy.stats import randint, uniform
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)
from IPython.display import display, Markdown

X_train = pd.read_pickle('../data/processed/tree_ready/X_train.pkl')
X_test = pd.read_pickle('../data/processed/tree_ready/X_test.pkl')
y_train = pd.read_pickle('../data/processed/tree_ready/y_train.pkl')
y_test = pd.read_pickle('../data/processed/tree_ready/y_test.pkl')

X_train_scaled = pd.read_pickle('../data/processed/neural_ready/X_train.pkl')
X_test_scaled = pd.read_pickle('../data/processed/neural_ready/X_test.pkl')

with open('../models/ml/random_forest.pkl', 'rb') as f:
    best_rf = pickle.load(f)
with open('../models/ml/xgboost.pkl', 'rb') as f:
    best_xgb_clean = pickle.load(f)
with open('../models/ml/lightgbm.pkl', 'rb') as f:
    best_lgbm = pickle.load(f)
with open('../models/ml/svm.pkl', 'rb') as f:
    svm = pickle.load(f)

with open('../data/interim/variable_labels.json') as f:
    variable_labels = json.load(f)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
interpretation_log = []

def evaluate_and_interpret(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
    results.append({'model': name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'auc': auc})
    md = f"### {name}\nAccuracy: {acc:.3f} | Precision: {prec:.3f} | Recall: {rec:.3f} | F1: {f1:.3f} | AUC: {auc:.3f}"
    display(Markdown(md))
    interpretation_log.append(md)

In [4]:
def search_labels(keyword):
    return {k: v for k, v in variable_labels.items() if keyword.lower() in v.lower()}

for kw in ['diet', 'food', 'iron', 'folic', 'fever', 'malaria', 'illness', 'meal', 'vegetable', 'meat', 'supplement']:
    result = search_labels(kw)
    if result:
        print(f"--- '{kw}' ---")
        for col, label in result.items():
            print(f"{col}: {label}")
        print()

--- 'food' ---
m39a_1: did eat any solid, semi-solid or soft foods yesterday
m39a_2: did eat any solid, semi-solid or soft foods yesterday
m39a_3: did eat any solid, semi-solid or soft foods yesterday
m39a_4: did eat any solid, semi-solid or soft foods yesterday
m39a_5: did eat any solid, semi-solid or soft foods yesterday
m39a_6: did eat any solid, semi-solid or soft foods yesterday
m39_1: number of times ate solid, semi-solid or soft food yesterday
m39_2: number of times ate solid, semi-solid or soft food yesterday
m39_3: number of times ate solid, semi-solid or soft food yesterday
m39_4: number of times ate solid, semi-solid or soft food yesterday
m39_5: number of times ate solid, semi-solid or soft food yesterday
m39_6: number of times ate solid, semi-solid or soft food yesterday
m42g_1: during pregnancy: talk about foods to eat
m42g_2: during pregnancy: talk about foods to eat
m42g_3: during pregnancy: talk about foods to eat
m42g_4: during pregnancy: talk about foods to eat
m42g_

In [6]:
rural_df = pd.read_pickle('../data/interim/rural_ir.pkl')

candidate_new_vars = ['v_XXX', 'v_YYY']  # fill in from Cell 2's output

for col in candidate_new_vars:
    if col in rural_df.columns:
        print(f"{col}: {rural_df[col].isnull().mean()*100:.1f}% missing")
    else:
        print(f"{col}: not found")

v_XXX: not found
v_YYY: not found


In [ ]:
with open('../data/processed/tree_ready/label_encoders.pkl', 'rb') as f:
    encoders = pickle.load(f)

tree_X_train_v2 = X_train.copy()
tree_X_test_v2 = X_test.copy()

# Recover the real province categories, then build one interaction column
# per province — instead of one meaningless "altitude * arbitrary_code" column
province_train = encoders['v024'].inverse_transform(X_train['v024'])
province_test = encoders['v024'].inverse_transform(X_test['v024'])

prov_dummies_train = pd.get_dummies(province_train, prefix='province')
prov_dummies_test = pd.get_dummies(province_test, prefix='province').reindex(
    columns=prov_dummies_train.columns, fill_value=0
)

for col in prov_dummies_train.columns:
    tree_X_train_v2[f'{col}_x_altitude'] = prov_dummies_train[col].values * X_train['v040'].values
    tree_X_test_v2[f'{col}_x_altitude'] = prov_dummies_test[col].values * X_test['v040'].values

# Confirm v190a's LabelEncoder order actually matches poorest->richest before trusting this:
print("v190a category order:", encoders['v190a'].classes_)
tree_X_train_v2['wealth_education_interact'] = tree_X_train_v2['v190a'] * tree_X_train_v2['v133']
tree_X_test_v2['wealth_education_interact'] = tree_X_test_v2['v190a'] * tree_X_test_v2['v133']

(2696, 14)


In [9]:
xgb_v2 = XGBClassifier(**best_xgb_clean.get_params())
xgb_v2.fit(tree_X_train_v2, y_train)
evaluate_and_interpret('XGBoost + Interactions', xgb_v2, tree_X_test_v2, y_test)

### XGBoost + Interactions
Accuracy: 0.707 | Precision: 0.536 | Recall: 0.521 | F1: 0.529 | AUC: 0.690

In [11]:
skf_10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

xgb_param_dist = {
    'max_depth': randint(3, 12),
    'learning_rate': uniform(0.01, 0.29),
    'n_estimators': randint(100, 600),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 10)
}

xgb_random = RandomizedSearchCV(
    XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss'),
    xgb_param_dist, n_iter=50, cv=skf_10, scoring='roc_auc', n_jobs=-1, random_state=42
)
xgb_random.fit(tree_X_train_v2, y_train)

print("Best params:", xgb_random.best_params_)
print("Best CV AUC:", xgb_random.best_score_)

evaluate_and_interpret('XGBoost (RandomizedSearch, wider tuning)', xgb_random.best_estimator_, tree_X_test_v2, y_test)

Best params: {'colsample_bytree': np.float64(0.713936197750987), 'learning_rate': np.float64(0.02069721473281451), 'max_depth': 3, 'min_child_weight': 2, 'n_estimators': 229, 'subsample': np.float64(0.7644148053272926)}
Best CV AUC: 0.7077234978242383


### XGBoost (RandomizedSearch, wider tuning)
Accuracy: 0.707 | Precision: 0.535 | Recall: 0.535 | F1: 0.535 | AUC: 0.707

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42))
])

base_learners_v2 = [
    ('rf', best_rf),
    ('xgb', best_xgb_clean),
    ('lgbm', best_lgbm),
    ('svm', svm_pipeline)     # scales internally, still receives tree_ready X like the others
]

meta_learner_xgb = XGBClassifier(n_estimators=100, max_depth=3, random_state=42, eval_metric='logloss')

stacking_v2 = StackingClassifier(
    estimators=base_learners_v2,
    final_estimator=meta_learner_xgb,
    cv=skf,
    n_jobs=-1
)

stacking_v2.fit(X_train, y_train)   # tree_ready — not X_train_scaled
evaluate_and_interpret('Stacking v2 (RF+XGB+LGBM+SVM, XGB meta)', stacking_v2, X_test, y_test)

### Stacking v2 (RF+XGB+LGBM+SVM, XGB meta)
Accuracy: 0.687 | Precision: 0.507 | Recall: 0.324 | F1: 0.395 | AUC: 0.670

In [13]:
calibrated_svm = CalibratedClassifierCV(svm, method='isotonic', cv=skf)
calibrated_svm.fit(X_train_scaled, y_train)
evaluate_and_interpret('SVM (Calibrated)', calibrated_svm, X_test_scaled, y_test)

### SVM (Calibrated)
Accuracy: 0.717 | Precision: 0.570 | Recall: 0.423 | F1: 0.485 | AUC: 0.685

In [1]:
results_df_v2 = pd.DataFrame(results)
print(results_df_v2.sort_values('auc', ascending=False))

results_df_v2.to_csv('../results/metrics/performance_improvement_experiments.csv', index=False)

NameError: name 'pd' is not defined

In [ ]:
candidate_new_vars = []
for kw in ['diet', 'food', 'iron', 'folic', 'fever', 'malaria', 'illness', 'meal', 'vegetable', 'meat', 'supplement']:
    candidate_new_vars.extend(search_labels(kw).keys())
candidate_new_vars = sorted(set(candidate_new_vars))
print(f"Found {len(candidate_new_vars)} candidate columns")

rural_df = pd.read_pickle('../data/interim/rural_ir.pkl')

usable_vars = []
for col in candidate_new_vars:
    if col in rural_df.columns:
        missing = rural_df[col].isnull().mean() * 100
        print(f"{col} ({variable_labels.get(col, '?')}): {missing:.1f}% missing")
        if missing < 30:   # adjust this cutoff to whatever you're comfortable imputing
            usable_vars.append(col)
    else:
        print(f"{col}: not found")

print("\nUsable candidates:", usable_vars)